# ML Lab 6

**Aim:** Build K-Nearest Neighbour and SVM classification models and evaluate their performance using appropriate metrics.

## Experiment 1: KNN on Breast Cancer Dataset
1. Load dataset and split into training/testing sets.
2. Train KNN classifier.
3. Predict class labels.
4. Evaluate with Accuracy, Confusion Matrix, Classification Report, Precision-Recall curve.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    roc_curve,
    auc
)

In [ ]:
# Load dataset
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')

print('Feature shape:', X.shape)
print('Target shape:', y.shape)
print('Class labels:', dict(enumerate(data.target_names)))
print('Class distribution:\n', y.value_counts())

In [ ]:
# Train-test split (80:20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('X_train:', X_train.shape, '| X_test:', X_test.shape)
print('y_train:', y_train.shape, '| y_test:', y_test.shape)

In [ ]:
# KNN model
knn_model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
knn_model.fit(X_train, y_train)

y_pred_knn = knn_model.predict(X_test)
y_prob_knn = knn_model.predict_proba(X_test)[:, 1]

acc_knn_train = accuracy_score(y_train, knn_model.predict(X_train))
acc_knn_test = accuracy_score(y_test, y_pred_knn)
cm_knn = confusion_matrix(y_test, y_pred_knn)

print('KNN Train Accuracy:', round(acc_knn_train, 4))
print('KNN Test Accuracy :', round(acc_knn_test, 4))
print('\nKNN Confusion Matrix:\n', cm_knn)
print('\nKNN Classification Report:\n')
print(classification_report(y_test, y_pred_knn, target_names=data.target_names))

In [ ]:
# KNN Precision-Recall curve
precision_knn, recall_knn, _ = precision_recall_curve(y_test, y_prob_knn)

plt.figure(figsize=(7, 5))
plt.plot(recall_knn, precision_knn, color='navy', label='KNN')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve (KNN)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Experiment 2: SVM on Breast Cancer Dataset
1. Train SVM classifier.
2. Predict class labels.
3. Evaluate with Accuracy, Confusion Matrix, Classification Report, Precision-Recall curve.
4. Visualize classifier behavior (2D PCA decision regions).

In [ ]:
# SVM model
svm_model = make_pipeline(StandardScaler(), SVC(kernel='rbf', probability=True, random_state=42))
svm_model.fit(X_train, y_train)

y_pred_svm = svm_model.predict(X_test)
y_prob_svm = svm_model.predict_proba(X_test)[:, 1]

acc_svm_train = accuracy_score(y_train, svm_model.predict(X_train))
acc_svm_test = accuracy_score(y_test, y_pred_svm)
cm_svm = confusion_matrix(y_test, y_pred_svm)

print('SVM Train Accuracy:', round(acc_svm_train, 4))
print('SVM Test Accuracy :', round(acc_svm_test, 4))
print('\nSVM Confusion Matrix:\n', cm_svm)
print('\nSVM Classification Report:\n')
print(classification_report(y_test, y_pred_svm, target_names=data.target_names))

In [ ]:
# SVM Precision-Recall curve
precision_svm, recall_svm, _ = precision_recall_curve(y_test, y_prob_svm)

plt.figure(figsize=(7, 5))
plt.plot(recall_svm, precision_svm, color='darkgreen', label='SVM')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve (SVM)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# SVM visualization using 2D PCA projection
X_train_scaled = StandardScaler().fit_transform(X_train)
pca = PCA(n_components=2, random_state=42)
X_train_2d = pca.fit_transform(X_train_scaled)

svm_2d = SVC(kernel='rbf', probability=True, random_state=42)
svm_2d.fit(X_train_2d, y_train)

x_min, x_max = X_train_2d[:, 0].min() - 1, X_train_2d[:, 0].max() + 1
y_min, y_max = X_train_2d[:, 1].min() - 1, X_train_2d[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 250), np.linspace(y_min, y_max, 250))
grid = np.c_[xx.ravel(), yy.ravel()]
zz = svm_2d.predict(grid).reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, zz, alpha=0.25, cmap='coolwarm')
sns.scatterplot(x=X_train_2d[:, 0], y=X_train_2d[:, 1], hue=y_train, palette='coolwarm', s=40)
plt.title('SVM Decision Regions (PCA 2D Projection)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend(title='Class')
plt.show()

## Experiment 3: Compare KNN vs SVM
1. Bar chart comparing train/test accuracy.
2. ROC curve comparison.
3. Side-by-side confusion matrix heatmaps.

In [ ]:
# 1) Accuracy bar chart
models = ['KNN', 'SVM']
train_acc = [acc_knn_train, acc_svm_train]
test_acc = [acc_knn_test, acc_svm_test]

x = np.arange(len(models))
width = 0.35

plt.figure(figsize=(8, 5))
plt.bar(x - width / 2, train_acc, width, label='Training Accuracy')
plt.bar(x + width / 2, test_acc, width, label='Testing Accuracy')
plt.xticks(x, models)
plt.ylim(0.0, 1.05)
plt.ylabel('Accuracy')
plt.title('KNN vs SVM: Training and Testing Accuracy')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
# 2) ROC comparison
fpr_knn, tpr_knn, _ = roc_curve(y_test, y_prob_knn)
fpr_svm, tpr_svm, _ = roc_curve(y_test, y_prob_svm)
auc_knn = auc(fpr_knn, tpr_knn)
auc_svm = auc(fpr_svm, tpr_svm)

plt.figure(figsize=(7, 5))
plt.plot(fpr_knn, tpr_knn, label=f'KNN (AUC={auc_knn:.3f})', color='navy')
plt.plot(fpr_svm, tpr_svm, label=f'SVM (AUC={auc_svm:.3f})', color='darkgreen')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison (KNN vs SVM)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# 3) Side-by-side confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('KNN Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Greens', ax=axes[1])
axes[1].set_title('SVM Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
comparison_df = pd.DataFrame({
    'Model': ['KNN', 'SVM'],
    'Train Accuracy': [acc_knn_train, acc_svm_train],
    'Test Accuracy': [acc_knn_test, acc_svm_test],
    'ROC-AUC': [auc_knn, auc_svm]
})
print(comparison_df)

## Answers to Questions

In [ ]:
# Recall for malignant class (class 0)
report_knn = classification_report(y_test, y_pred_knn, output_dict=True)
report_svm = classification_report(y_test, y_pred_svm, output_dict=True)

recall_malignant_knn = report_knn['0']['recall']
recall_malignant_svm = report_svm['0']['recall']

higher_accuracy_model = 'KNN' if acc_knn_test > acc_svm_test else ('SVM' if acc_svm_test > acc_knn_test else 'Tie')
better_recall_model = 'KNN' if recall_malignant_knn > recall_malignant_svm else ('SVM' if recall_malignant_svm > recall_malignant_knn else 'Tie')

gap_knn = acc_knn_train - acc_knn_test
gap_svm = acc_svm_train - acc_svm_test
overfit_model = 'KNN' if gap_knn > gap_svm else ('SVM' if gap_svm > gap_knn else 'Similar')

print('1. Which model achieved higher accuracy?')
print('  ', higher_accuracy_model)
print('\n2. Which model showed better recall for malignant cases?')
print('  ', better_recall_model)
print('\n3. Which model is more prone to overfitting and why?')
print('  ', overfit_model, 'shows a larger train-test accuracy gap.')
print('\n4. Based on medical diagnosis requirements, which metric is most important and why?')
print('   Recall is most important because false negatives (missed malignant cases) are critical.')